## Load dataset

In [1]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

newsgroups = fetch_20newsgroups(
    subset="train",
    categories=["sci.space", "sci.med"],
    remove=("headers", "footers", "quotes"),
)

df = pd.DataFrame({"text": newsgroups.data, "category": newsgroups.target})
df = df.dropna().loc[df["text"].str.len() > 20].head(500)
print(f"Loaded {len(df)} articles.")
df.head()

Loaded 500 articles.


,text,category
0,"From: ""Phil G. Fraering"" <pgf@srl03.cacs.usl.e...",1
1,"\nIt isn't that rare, actually. Many cases th...",0
2,"I have between 15 and 25 nosebleeds each week,...",0
3,\nIt depends on the cause of the pneumonia. F...,0
4,Is there a few Grasp pictures of space related...,1


In [2]:
print((newsgroups.DESCR))

.. _20newsgroups_dataset:

The 20 newsgroups text dataset
------------------------------

The 20 newsgroups dataset comprises around 18000 newsgroups posts on
20 topics split in two subsets: one for training (or development)
and the other one for testing (or for performance evaluation). The split
between the train and test set is based upon a messages posted before
and after a specific date.

This module contains two loaders. The first one,
:func:`sklearn.datasets.fetch_20newsgroups`,
returns a list of the raw texts that can be fed to text feature
extractors such as :class:`~sklearn.feature_extraction.text.CountVectorizer`
with custom parameters so as to extract feature vectors.
The second one, :func:`sklearn.datasets.fetch_20newsgroups_vectorized`,
returns ready-to-use features, i.e., it is not necessary to use a feature
extractor.

**Data Set Characteristics:**

=================   ==========
Classes                     20
Samples total            18846
Dimensionality               1

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 500 entries, 0 to 521
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   text      500 non-null    str  
 1   category  500 non-null    int64
dtypes: int64(1), str(1)
memory usage: 655.6 KB


## Preprocess

In [4]:
import re
import spacy
import contractions
from spellchecker import SpellChecker
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords

from tqdm import tqdm

nlp = spacy.load("en_core_web_sm")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stopwords = set(stopwords.words("english"))
spell = SpellChecker()

def preprocess(text):

    # Contractions
    text = contractions.fix(text)

    # Case normalization, special characters, extra white space
    clean_text = text.lower()
    clean_text = re.sub(r"[^a-z\s]", "", clean_text)
    clean_text = re.sub(r"\s+", " ", clean_text).strip()

    # spell correction
    corrected_words = []
    for word in clean_text.split():
        corrected_spell = spell.correction(word)

        if corrected_spell is not None:
            corrected_words.append(corrected_spell)
        else:
            corrected_words.append(word)
    corrected_text = " ".join(corrected_words)

    # tokenization
    tokens = word_tokenize(clean_text)

    # stopwords, stemming, lemmatizing
    stem_tokens = [stemmer.stem(w) for w in tokens if w not in stopwords]
    lemmatize_tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stopwords]

    return {"stemmed": " ".join(stem_tokens), "lemmatized": " ".join(lemmatize_tokens)}

tqdm.pandas(desc="Preprocessing Text")
processed = df.text.progress_apply(preprocess)
df["stemmed"] = processed.apply(lambda x:x["stemmed"])
df["lemmatized"] = processed.apply(lambda x:x["lemmatized"])
df.head()

Preprocessing Text: 100%|██████████| 500/500 [39:34<00:00,  4.75s/it]   


,text,category,stemmed,lemmatized
0,"From: ""Phil G. Fraering"" <pgf@srl03.cacs.usl.e...",1,phil g fraer pgfsrlcacsusledu right profit cas...,phil g fraering pgfsrlcacsusledu right profiti...
1,"\nIt isn't that rare, actually. Many cases th...",0,rare actual mani case call parkinson diseas tu...,rare actually many case called parkinson disea...
2,"I have between 15 and 25 nosebleeds each week,...",0,noseble week result genet predisposit weak cap...,nosebleed week result genetic predisposition w...
3,\nIt depends on the cause of the pneumonia. F...,0,depend pneumonia treat bacteri pneumonia young...,depends pneumonia treating bacterial pneumonia...
4,Is there a few Grasp pictures of space related...,1,grasp pictur space relat item name space stati...,grasp picture space related item namely space ...


## Tf-IDF

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=1000)
tfidf = tfidf_vectorizer.fit_transform(df.lemmatized)



## Word2Vec (Skip-gram)

In [6]:
from gensim.models import Word2Vec

tokenized_sentences = [text.split() for text in df["lemmatized"]]

# Skip-gram model : sg=1
# CBOW : sg=0
w2v_model = Word2Vec(
    sentences=tokenized_sentences, vector_size=100, window=5, min_count=2, sg=1
)

print("Vocabulary Size:", len(w2v_model.wv))
if "space" in w2v_model.wv:
    print("\nMost similar to 'space':")
    for word, score in w2v_model.wv.most_similar("space", topn=3):
        print(f" - {word}: {score:.4f}")

Vocabulary Size: 5582

Most similar to 'space':
 - shuttle: 0.9897
 - program: 0.9849
 - nasa: 0.9795


## Word2Vec (CBOW)

In [7]:
from gensim.models import Word2Vec

tokenized_sentences = [text.split() for text in df.lemmatized]

w2v_model = Word2Vec(sentences=tokenized_sentences, min_count=2)

if "space" in w2v_model.wv:
    print("Most similar to word 'space': ")
    for word, score in w2v_model.wv.most_similar("space",topn=3):
        print(f"- {word}: {score}")

Most similar to word 'space': 
- program: 0.9997329115867615
- would: 0.9997289180755615
- one: 0.9997119307518005


## GloVe

In [8]:
import gensim.downloader as api

glove_model = api.load("glove-wiki-gigaword-50")

print("Most similar to 'space':")
for word, score in glove_model.most_similar("space", topn=3):
    print(f" - {word}: {score:.4f}")

Most similar to 'space':
 - earth: 0.7599
 - shuttle: 0.7392
 - orbit: 0.7259
